In [1]:
from rdflib import Graph, RDF, RDFS, SH
import json

# -- URL for DCAT-US 3 SHACL --
DCAT_US_3_SHACL_URL = (
    "https://raw.githubusercontent.com/DOI-DO/dcat-us/main/shacl/dcat-us_3.0_shacl_shapes.ttl"
)

# -- Allowed namespaces for REAL DCAT metadata classes --
VALID_CLASS_PREFIXES = [
    "http://www.w3.org/ns/dcat#",
    "http://purl.org/dc/terms/",
    "http://xmlns.com/foaf/0.1/",
    "http://www.w3.org/ns/locn#",
    "http://www.w3.org/2006/vcard/ns#",
]

# -- Expected class-level requirements for DCAT-US --
CLASS_USAGE = {
    "http://www.w3.org/ns/dcat#Catalog": {
        "requirement": "mandatory",
        "min_instances": 1,
        "max_instances": 1,
    },
    "http://www.w3.org/ns/dcat#Dataset": {
        "requirement": "mandatory",
        "min_instances": 1,
        "max_instances": None,
    },
    "http://www.w3.org/ns/dcat#Distribution": {
        "requirement": "recommended_if_applicable",
        "min_instances": 0,
        "max_instances": None,
    },
    "http://xmlns.com/foaf/0.1/Agent": {
        "requirement": "mandatory_if_applicable",
        "min_instances": 0,
        "max_instances": None,
    },
    "http://www.w3.org/2006/vcard/ns#Kind": {
        "requirement": "mandatory_if_applicable",
        "min_instances": 0,
        "max_instances": None,
    },
    "http://purl.org/dc/terms/PeriodOfTime": {
        "requirement": "optional",
        "min_instances": 0,
        "max_instances": None,
    },
    "http://www.w3.org/ns/locn#Geometry": {
        "requirement": "optional",
        "min_instances": 0,
        "max_instances": None,
    },
}

def build_model():
    print("Loading DCAT-US 3 SHACL...")
    g = Graph()
    g.parse(DCAT_US_3_SHACL_URL, format="turtle")
    print(f"Loaded {len(g)} triples")

    model = {}

    # Find SHACL NodeShapes
    for shape in g.subjects(RDF.type, SH.NodeShape):

        # Extract the target class
        targets = list(g.objects(shape, SH.targetClass))
        if not targets:
            continue

        target = str(targets[0])

        # Filter out helper shapes (shapes/dcat-us#XXX)
        if not any(target.startswith(prefix) for prefix in VALID_CLASS_PREFIXES):
            continue

        # Prepare class entry
        class_entry = {
            "shape": str(shape),
            "class_usage": CLASS_USAGE.get(
                target,
                {
                    "requirement": "optional",
                    "min_instances": 0,
                    "max_instances": None,
                },
            ),
            "mandatory": {},
            "recommended": {},
            "optional": {},
        }

        # Process property shapes
        for pshape in g.objects(shape, SH.property):
            prop = g.value(pshape, SH.path)
            if prop is None:
                continue
            prop = str(prop)

            # Determine constraints
            mincount = g.value(pshape, SH.minCount)
            maxcount = g.value(pshape, SH.maxCount)

            # Extract definitions/labels
            definition = g.value(pshape, SH.description) or g.value(pshape, RDFS.comment)
            if definition:
                definition = str(definition)

            label = g.value(pshape, RDFS.label)
            if label:
                label = str(label)

            constraint_entry = {}
            if definition:
                constraint_entry["definition"] = definition
            if label:
                constraint_entry["label"] = label
            if mincount:
                constraint_entry["minCount"] = int(mincount)
            if maxcount:
                constraint_entry["maxCount"] = int(maxcount)

            # Classification
            if mincount and int(mincount) >= 1:
                class_entry["mandatory"][prop] = constraint_entry
            else:
                # (Later we can distinguish recommended vs optional
                #  based on DCAT-US guidance; for now all non-mandatory are optional)
                class_entry["optional"][prop] = constraint_entry

        # Store class entry
        model[target] = class_entry

    return model


def main():
    model = build_model()

    with open("dcat_us_3_model.json", "w", encoding="utf-8") as f:
        json.dump(model, f, indent=2, ensure_ascii=False)

    print("\n✓ Wrote dcat_us_3_model.json")
    print("✓ Classes captured:", len(model))


if __name__ == "__main__":
    main()


Loading DCAT-US 3 SHACL...
Loaded 4042 triples

✓ Wrote dcat_us_3_model.json
✓ Classes captured: 20
